# 05 · Forward test

The evaluation pre-registered in [docs/forward_test_plan.md](../docs/forward_test_plan.md): the models of 02–04 and Timestral (06), frozen, scored on 1 November 2026 to 30 April 2027, a period none of them and none of the hypotheses can reflect.

This notebook was written and committed **before the period began, without being run**; its cells fix the analysis as the plan fixes the hypotheses. It is run once all outcomes are known. Anything that has to change after that (a source that moved, a model file that cannot be used, an error in this notebook) is listed under *Deviations* with its reason, and where possible the result is shown both ways.

Sections: frozen models · data · forecasts · the nine hypotheses · exploratory results (TimesFM's covariates, headline readers, prediction markets) · deviations · conclusions.

## Setup

In [ ]:
import hashlib
import json
import os
from functools import partial
from importlib.metadata import distribution, version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sbench import markets, plots, tsfm
from sbench.backtest import decision_points, load_results, run_forecast_backtest
from sbench.context import NewsFeed, SearchInterest, covariates
from sbench.data import (
    GOOGLE_TRENDS_DAY,
    Series,
    cached,
    fetch_polymarket_closed_events,
    load_google_trends,
    load_polymarket_history,
)
from sbench.elicitation import DirectionChoice, ThresholdNoul
from sbench.forecast import make_task
from sbench.headlines import HeadlineReader, read_all
from sbench.hypotheses import paired_bootstrap, sd_difference, slope_per_sd, verdict
from sbench.panel import ASSET_CLASS, download_panel
from sbench.predictors import DecisionModelPredictor, EmpiricalPredictor, RandomWalkPredictor
from sbench.scoring import per_forecast, rank_ic, skill_table
from sbench.states import build_indicator_state, build_news_state, build_price_state, build_search_state
from sbench.tsfm import QuantilePredictor, TimesFM3, Timestral

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # weights are cached; never re-check the Hub

PERIODS = {"first": ("2026-11-01", "2027-04-30"),    # the evaluation
           "update": ("2026-11-01", "2027-10-31")}   # repeated over 12 months for the low-power H6 and H7
PERIOD = "first"
START, END = PERIODS[PERIOD]

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SNAPSHOTS = ROOT / "data" / "snapshots" / "05_forward_test" / PERIOD   # downloaded once; committed
RESULTS = ROOT / "data" / "results" / "05_forward_test" / PERIOD       # resumable; committed
CHARTS = ROOT / "outputs" / "05_forward_test" / PERIOD                 # regenerated; not committed
RAW_NEWS = ROOT / "data" / "raw" / "gkg"                               # python -m sbench.news; not committed

HISTORY_START = "2023-01-01"     # as in 02 and 03: the context the models had there
NEWS_START = "2026-10-01"        # 30 days of news baseline before the first decision point
TRENDS_START = "2026-09-01"      # search-interest baseline, as in 03
HORIZON, STEP = 5, 5             # horizon 5 on every fifth observation
MIN_HISTORY = 100
STATE_LOOKBACK, RELATED_LOOKBACK = 60, 20

plots.use_style()
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## Frozen models

The plan identifies each decision model by the SHA-256 of its file, TimesFM by its Hugging Face revision and package version, and Timestral by its revision and the package's commit. A different file is a different model, not part of this test, so the notebook stops rather than run one.

In [ ]:
MODEL_DIR = Path(os.environ.get("SBENCH_MODEL_DIR", Path.home() / ".lmstudio" / "models"))
FROZEN = {   # model: (file under MODEL_DIR, SHA-256), from the plan
    "clef": ("ggml-org/Clef-GGUF/Clef-Q4_K_M.gguf", "9997675e4c9e9016273372b5fdedbdda7ad14285bc722949e4b9daca62bf3822"),
    "lev": ("ggml-org/lev-GGUF/lev-Q4_K_M.gguf", "3f61b27c00a098dbc79ed099cca3985cbe0b63d17ac7fedebf2dca1d84f7f3a8"),
    "kev-9b": ("ggml-org/Kev-9B-GGUF/Kev-9B-Q4_K_M.gguf", "86a6084984a6ef6818eb12c07cdc67ed3c47b00208d41a8816c64301bc6bcbda"),
}
TIMESFM_REPO, TIMESFM_REVISION, TIMESFM_PACKAGE = (
    "google/timesfm-3.0-pytorch", "43046b85ec22d584a13f8098c2ed39c889e129c2", "3.0.2")
TIMESTRAL_REVISION, TIMESTRAL_COMMIT = "9874d252ab0aae16f7f7c999f98f98d6ad7d8365", "a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 24):
            h.update(chunk)
    return h.hexdigest()


def package_commit(dist: str) -> str:
    """The git commit a package was installed from (pip's direct_url.json)."""
    info = json.loads(distribution(dist).read_text("direct_url.json") or "{}")
    return info.get("vcs_info", {}).get("commit_id", "not from git")


def hub_revision(repo: str) -> str:
    from huggingface_hub.constants import HF_HUB_CACHE
    ref = Path(HF_HUB_CACHE) / f"models--{repo.replace('/', '--')}" / "refs" / "main"
    return ref.read_text().strip() if ref.exists() else "not in cache"


checks = [{"model": m, "expected": h[:12], "found": sha256(MODEL_DIR / f)[:12] if (MODEL_DIR / f).exists() else "missing"}
          for m, (f, h) in FROZEN.items()]
checks += [{"model": "timesfm3 revision", "expected": TIMESFM_REVISION[:12], "found": hub_revision(TIMESFM_REPO)[:12]},
           {"model": "timesfm package", "expected": TIMESFM_PACKAGE, "found": version("timesfm")},
           {"model": "timestral revision", "expected": TIMESTRAL_REVISION[:12], "found": Timestral().revision[:12]},
           {"model": "timestral package", "expected": TIMESTRAL_COMMIT[:12], "found": package_commit("timestral")[:12]}]
checks = pd.DataFrame(checks).assign(ok=lambda d: d["expected"] == d["found"]).set_index("model")
display(checks)
assert checks["ok"].all(), "a model differs from the plan's: report it as a deviation instead of running past it"

## Data

The same eight instruments, sources and close times as in 03, downloaded afresh from 2023 (the context TimesFM and the decision models had in 02 and 03) to the end of the period. The last decision point of each horizon is the last one whose outcome falls inside the period.

In [ ]:
SERIES = download_panel(SNAPSHOTS / "prices", HISTORY_START, END)
BY_KEY = {s.key: s for s in SERIES}
panel = pd.DataFrame([{"series": s.key, "class": s.meta["asset_class"], "rows": len(s),
                       "first": s.df["t"].min().date(), "last": s.df["t"].max().date(),
                       "max gap (days)": s.df["t"].diff().max().days} for s in SERIES]).set_index("series")
display(panel)
assert (pd.to_datetime(panel["last"]) >= pd.Timestamp(END) - pd.Timedelta(days=4)).all(), \
    "a snapshot ends before the period: it was downloaded too early; delete it and rerun"

IDXS = {
    1: {s.key: decision_points(s, 1, MIN_HISTORY, 1, start=START) for s in SERIES},
    HORIZON: {s.key: decision_points(s, HORIZON, MIN_HISTORY, STEP, start=START) for s in SERIES},
}
pd.DataFrame({f"horizon {h}": {k: len(v) for k, v in idxs.items()} for h, idxs in IDXS.items()}).T.assign(
    total=lambda d: d.sum(axis=1))

### News and search interest

News comes from GDELT's raw files, collected for the period with `python -m sbench.news 2026-10-01 2027-04-30 data/raw/gkg` (the update needs the files to 31 October 2027); only the headlines shown at the decision points are kept in the committed snapshot. Search interest is downloaded from Google Trends as in 03 (about 30 requests, spaced 30 s apart). News, search terms and the related series per instrument are those of 03.

In [ ]:
NEWS_LABELS = {"BTC": "Bitcoin", "ETH": "Ether", "SPY": "US stocks", "AAPL": "Apple", "GLD": "gold",
               "USO": "oil", "EURUSD": "the euro", "USDJPY": "the yen"}
NEWS_DIR = SNAPSHOTS / "news"
if not (NEWS_DIR / "news_coverage.csv.gz").exists():
    full_feed = NewsFeed.from_raw(RAW_NEWS, NEWS_START, END)
    asked = [(s.key, pd.Timestamp(s.available_at()[i])) for s in SERIES for i in {*IDXS[1][s.key], *IDXS[HORIZON][s.key]}]
    full_feed.restrict_to(asked).save(NEWS_DIR)
FEED = NewsFeed.load(NEWS_DIR, labels=NEWS_LABELS)

TRENDS_TERMS = {"BTC": ("btc", 0), "ETH": ("eth", 0), "SPY": ("s&p 500", 0), "AAPL": ("aapl", 0),
                "GLD": ("gold price", 0), "USO": ("crude oil", 0), "EURUSD": ("eur usd", 0), "USDJPY": ("yen", 7)}
SEARCH = SearchInterest(
    {k: Series(k, f"Google searches for '{term}'",
               cached(SNAPSHOTS / f"trends_{k}.csv", load_google_trends, term, TRENDS_START, END, cat=cat, pause=30),
               kind="covariate", close_time=GOOGLE_TRENDS_DAY)
     for k, (term, cat) in TRENDS_TERMS.items()},
    {k: term + (" (finance category)" if cat else "") for k, (term, cat) in TRENDS_TERMS.items()})

RELATED = {
    "BTC": ("ETH", "SPY"), "ETH": ("BTC", "SPY"), "SPY": ("AAPL", "USDJPY"), "AAPL": ("SPY", "USDJPY"),
    "GLD": ("EURUSD", "USDJPY"), "USO": ("SPY", "EURUSD"), "EURUSD": ("USDJPY", "GLD"), "USDJPY": ("EURUSD", "SPY"),
}
INPUTS = {   # TimesFM covariates per variant and instrument (decision models read the same through their state)
    "related": {s.key: tuple(BY_KEY[k] for k in RELATED[s.key]) for s in SERIES},
    "news": {s.key: covariates(s, partial(FEED.features, s.key), prefix="news_") for s in SERIES},
    "search": {s.key: covariates(s, partial(SEARCH.features, s.key)) for s in SERIES},
}
STATES = {
    "price": partial(build_price_state, lookback=STATE_LOOKBACK),
    "indicators": partial(build_indicator_state, lookback=STATE_LOOKBACK),
    "news": partial(build_news_state, feed=FEED, lookback=STATE_LOOKBACK),
    "search": partial(build_search_state, interest=SEARCH, lookback=STATE_LOOKBACK),
}

KNOWN_AT = {s.key: dict(zip(s.df["t"], s.available_at())) for s in SERIES}
pd.DataFrame([{
    "series": s.key,
    **{f"news, h={h}": np.mean([FEED.coverage_at(s.key, s.available_at()[i]) is not None for i in IDXS[h][s.key]])
       for h in IDXS},
    **{f"search, h={h}": np.mean([SEARCH.ratios(s.key, s.available_at()[i]) is not None for i in IDXS[h][s.key]])
       for h in IDXS},
} for s in SERIES]).set_index("series").rename(columns=lambda c: f"{c} available")

## Forecasts

The predictors and variants the hypotheses and the plan's reports need, each as in 03: decision models in their formats from 01 with the named state of the last 60 closes, TimesFM 3.0 and Timestral with and without covariates, and the two baselines. Results are appended after each forecast, so the run resumes where it stopped; a model the server cannot load is reported and skipped until it can be.

In [ ]:
FORMATS = {"clef": ThresholdNoul(), "lev": DirectionChoice(include_flat=False), "kev-9b": ThresholdNoul()}
RUNS = {   # results file: (horizon, variant, decision models)
    "price_h1": (1, "price", ("clef", "lev")),
    "news_h1": (1, "news", ("clef", "lev")),
    "search_h1": (1, "search", ("clef", "lev")),
    "price": (HORIZON, "price", ("clef", "lev", "kev-9b")),
    "related": (HORIZON, "related", ()),
    "indicators": (HORIZON, "indicators", ("clef", "lev", "kev-9b")),
    "news": (HORIZON, "news", ("clef", "lev")),
    "search": (HORIZON, "search", ("clef", "lev")),
}


def references(variant: str) -> list:
    if variant == "price":
        return [RandomWalkPredictor(), EmpiricalPredictor(), QuantilePredictor(TimesFM3(), name="timesfm3", log_space=True),
                QuantilePredictor(Timestral(), name="timestral", log_space=True)]
    if variant in INPUTS:
        return [QuantilePredictor(TimesFM3(), name=f"timesfm3+{variant}", log_space=True, use_related=True),
                QuantilePredictor(Timestral(), name=f"timestral+{variant}", log_space=True, use_related=True)]
    return []


def decision(variant: str, model: str) -> DecisionModelPredictor:
    fmt = FORMATS[model]
    return DecisionModelPredictor(model, fmt, STATES[variant],
                                  name=f"{model}/{fmt.name}" + ("" if variant == "price" else f"+{variant}"))


def run(file: str, predictors: list) -> None:
    horizon, variant, _ = RUNS[file]
    for s in SERIES:
        run_forecast_backtest(s, predictors, horizon, RESULTS / f"{file}.csv", IDXS[horizon][s.key],
                              related=INPUTS.get(variant, {}).get(s.key, ()), verbose=False)


def available(model: str) -> bool:
    s = BY_KEY["GLD"]
    task = make_task(s, IDXS[1]["GLD"][-1], 1, related=INPUTS["news"]["GLD"])
    try:
        DecisionModelPredictor(model, ThresholdNoul(), STATES["news"]).predict(task)
        return True
    except Exception as e:  # noqa: BLE001 -- reported, model skipped
        print(f"{model}: unavailable, skipped ({' '.join(str(e).split())[:240]})")
        return False


for file, (_, variant, _) in RUNS.items():
    if references(variant):
        run(file, references(variant))
tsfm.release()   # the foundation models would otherwise hold GPU memory the model server needs
SERVED = [m for m in FORMATS if available(m)]
for m in SERVED:
    for file, (_, variant, models) in RUNS.items():
        if m in models:
            run(file, [decision(variant, m)])

frames = {f: load_results(RESULTS / f"{f}.csv") for f in RUNS}
print(f"decision models served: {', '.join(SERVED)}")
for f, (h, _, _) in RUNS.items():
    expected = pd.Series({k: len(v) for k, v in IDXS[h].items()})
    got = frames[f].groupby(["predictor", "series"])["t"].nunique().unstack().reindex(columns=expected.index)
    print(f"{f:11s}: {frames[f]['predictor'].nunique()} predictors; pairs with missing decision points: "
          f"{int(got.fillna(0).lt(expected, axis=1).to_numpy().sum())}")
results = pd.concat(frames.values(), ignore_index=True)

## The hypotheses

Every comparison is paired on decision points; skills use the plan's definition and a 95% bootstrap interval (2,000 resamples, seed 0). Decision points without news (archive gaps) are left out of every comparison that involves the news variant, for all predictors alike. A hypothesis **holds** when its interval lies entirely on the stated side of zero, **fails** when the point estimate is on the other side, and is **inconclusive** otherwise. H1 holds unless some predictor of 02 and 03 has an interval entirely above zero; its row shows the predictor closest to that (Timestral, added later, is tested by H9). H4 holds only if it holds for both models.

In [ ]:
per = per_forecast(results)
per["base"] = per["predictor"].str.replace(r"\+\w+$", "", regex=True)   # the same model on price only
per["variant"] = per["predictor"].str.extract(r"\+(\w+)$", expand=False).fillna("price")
per["asset_class"] = per["series"].map(ASSET_CLASS)
per["news_ok"] = [FEED.coverage_at(k, KNOWN_AT[k][t]) is not None for k, t in zip(per["series"], per["t"])]
print(f"decision points without news, left out of news comparisons: "
      f"{per.loc[per['predictor'] == 'random_walk', 'news_ok'].eq(False).sum()}")


def versus(df: pd.DataFrame, model: str, baseline: str, metric: str) -> dict:
    """Skill of model against baseline on the points both answered."""
    d = df[df["predictor"].isin([model, baseline])].dropna(subset=[metric])
    s = skill_table(d, metric, baseline=baseline)
    if s.empty:
        return {"estimate": np.nan, "lo": np.nan, "hi": np.nan, "N": 0}
    s = s.iloc[0]
    return {"estimate": s["skill"], "lo": s["lo"], "hi": s["hi"], "N": int(s["N"])}


h1, h5 = per[per["horizon"] == 1], per[per["horizon"] == HORIZON]
h1_news = h1[h1["news_ok"]]
rows = []

# H1: no predictor beats the random walk
vs_rw = []
for h, d in ((1, h1), (HORIZON, h5)):
    for metric in ("crps_n", "brier_up"):
        for p in d.loc[~d["predictor"].str.match(r"random_walk$|timestral"), "predictor"].unique():
            r = versus(d, p, "random_walk", metric)
            if r["N"]:
                vs_rw.append({"horizon": h, "metric": metric, "predictor": p, **r})
vs_rw = pd.DataFrame(vs_rw)
best = vs_rw.loc[vs_rw["lo"].idxmax()]   # the predictor closest to beating the random walk
rows.append({"hypothesis": "H1", "comparison": f"every predictor of 02 and 03 vs random walk (closest: {best['predictor']}, "
                                               f"horizon {best['horizon']})",
             "metric": best["metric"], **best[["estimate", "lo", "hi", "N"]].to_dict(),
             "verdict": "fails" if (vs_rw["lo"] > 0).any() else "holds"})

# H2, H3, H5, H6: a variant against the same model on price only
for name, model, baseline, metric, d, side in (
        ("H2", "lev/direction_choice+news", "lev/direction_choice", "brier_up", h1_news, "below"),
        ("H3", "clef/threshold_noul+news", "clef/threshold_noul", "crps_n", h1_news, "below"),
        ("H5", "clef/threshold_noul+search", "clef/threshold_noul", "brier_up", h1, "above"),
        ("H6", "kev-9b/threshold_noul+indicators", "kev-9b/threshold_noul", "crps_n", h5, "above")):
    r = versus(d, model, baseline, metric)
    rows.append({"hypothesis": name, "comparison": f"{model} vs {baseline}, horizon {d['horizon'].iloc[0]}",
                 "metric": metric, **r, "verdict": verdict(r["estimate"], r["lo"], r["hi"], side)})

# H4: news widens P(up), for both models
h4 = {}
for model in ("lev/direction_choice", "clef/threshold_noul"):
    p = h1_news[h1_news["predictor"].isin([model, f"{model}+news"])].pivot_table(
        index=["series", "t"], columns="predictor", values="p_up").dropna()
    h4[model] = paired_bootstrap(sd_difference, p.get(f"{model}+news", pd.Series(dtype=float)),
                                 p.get(model, pd.Series(dtype=float)))
h4_verdicts = [verdict(r["estimate"], r["lo"], r["hi"], "above") for r in h4.values()]
for model, r in h4.items():
    rows.append({"hypothesis": "H4", "comparison": f"sd of P(up), {model}+news minus price only, horizon 1",
                 "metric": "sd P(up)", **r,
                 "verdict": "holds" if all(v == "holds" for v in h4_verdicts)
                 else "fails" if "fails" in h4_verdicts else "inconclusive"})

# H7: prices revert from the Bollinger bands
pct_b = []
for s in SERIES:
    c = s.df["close"]
    mid, sd = c.rolling(20).mean(), c.rolling(20).std()
    pct_b.append(pd.DataFrame({"series": s.key, "t": s.df["t"], "pct_b": (c - (mid - 2 * sd)) / (4 * sd)}))
b = h5[h5["predictor"] == "random_walk"].merge(pd.concat(pct_b), on=["series", "t"])
r = paired_bootstrap(slope_per_sd, b["pct_b"], b["up"].astype(float))
rows.append({"hypothesis": "H7", "comparison": "realised up-rate on Bollinger %B, horizon 5, pooled",
             "metric": "slope per sd of %B", **r, "verdict": verdict(r["estimate"], r["lo"], r["hi"], "below")})

# H8, H9: Timestral against TimesFM 3.0 and the random walk (added after 06)
for name, baseline in (("H8", "timesfm3"), ("H9", "random_walk")):
    r = versus(h1, "timestral", baseline, "crps_n")
    rows.append({"hypothesis": name, "comparison": f"timestral vs {baseline}, horizon 1", "metric": "crps_n", **r,
                 "verdict": verdict(r["estimate"], r["lo"], r["hi"], "above")})

HYPOTHESES = pd.DataFrame(rows).sort_values("hypothesis", kind="stable").set_index("hypothesis")
HYPOTHESES

H1 in detail: every predictor against the random walk (the hypothesis fails if any interval lies entirely above zero).

In [ ]:
display(vs_rw.pivot_table(index="predictor", columns=["horizon", "metric"], values="estimate"))
plots.dot_plot(vs_rw.assign(label=lambda d: d["predictor"] + ", h=" + d["horizon"].astype(str)),
               y="label", x="estimate", hue="metric", lo="lo", hi="hi",
               xlabel="skill vs random walk (95% CI)", title="H1: every predictor against the random walk",
               path=CHARTS / "01_h1_vs_random_walk.png");

In [ ]:
effects = HYPOTHESES.drop(index="H1").reset_index()
effects["label"] = effects["hypothesis"] + ": " + effects["comparison"].str.split(",").str[0]
plots.dot_plot(effects, y="label", x="estimate", lo="lo", hi="hi",
               xlabel="effect (95% CI); each hypothesis states its side of zero",
               title="H2 to H9 in the forward period", path=CHARTS / "02_hypotheses.png");

## Exploratory results

Reported as estimates with intervals, without verdicts, as the plan states.

### Timestral at horizon 5, and the foundation models' covariates

In [ ]:
display(pd.DataFrame([{"comparison": f"timestral vs {b}", "metric": m, **versus(h5, "timestral", b, m)}
                      for b in ("timesfm3", "random_walk") for m in ("crps_n", "brier_up")]).set_index(["comparison", "metric"]))

rows = []
for model in ("timesfm3", "timestral"):
    for variant in INPUTS:
        for h, d in ((1, h1_news if variant == "news" else h1), (HORIZON, h5[h5["news_ok"]] if variant == "news" else h5)):
            for metric in ("crps_n", "brier_up"):
                r = versus(d, f"{model}+{variant}", model, metric)
                if r["N"]:
                    rows.append({"model": model, "variant": variant, "horizon": h, "metric": metric, **r})
pd.DataFrame(rows).set_index(["model", "variant", "horizon", "metric"])

### Headline readers

lev and clef rate each of an instrument's 15 newest headlines with the frozen question of `sbench.headlines`, at every decision point of both horizons. The news score is set against the next return; intensity against the size of the next move, within each instrument and beyond the past move and trailing volatility, as in 04.

In [ ]:
READERS = ["lev", "clef"]
asked = list(dict.fromkeys((s.key, pd.Timestamp(s.available_at()[i])) for h in IDXS for s in SERIES
                           for i in IDXS[h][s.key]))


def titles(key: str, until: pd.Timestamp) -> list[str]:
    return FEED.headlines_at(key, until, FEED.max_headlines)["title"].tolist()


scores = {}
for model in READERS:
    try:
        scores[model] = read_all(HeadlineReader(model), asked, titles, NEWS_LABELS,
                                 RESULTS / f"headline_reads_{model}.jsonl")
    except Exception as e:  # noqa: BLE001 -- a reader the server cannot load is reported and skipped
        print(f"{model}: skipped ({' '.join(str(e).split())[:240]})")


def signals(horizon: int, window: int = 60) -> pd.DataFrame:
    rows = []
    for s in SERIES:
        close = s.df["close"].to_numpy()
        vol = np.log(s.df["close"]).diff().rolling(window).std().to_numpy()
        for i in IDXS[horizon][s.key]:
            rows.append({"series": s.key, "t": s.df["t"].iloc[i], "until": pd.Timestamp(s.available_at()[i]),
                         "return": np.log(close[i + horizon] / close[i]),
                         "past size": abs(np.log(close[i] / close[i - horizon])), "trailing vol": vol[i]})
    d = pd.DataFrame(rows).assign(size=lambda d: d["return"].abs())
    for model, sc in scores.items():
        d = d.merge(sc[["series", "until", "score", "intensity"]].rename(
            columns={"score": f"{model} score", "intensity": f"{model} intensity"}), on=["series", "until"], how="left")
    return d


def unexplained_size(d: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Per instrument: percentile ranks, and the next move's size rank left after the past move's size and trailing volatility."""
    out = []
    for _, g in d.dropna(subset=["trailing vol"]).groupby("series"):
        g = g.assign(**{c: g[c].rank(pct=True) for c in [*cols, "size", "past size", "trailing vol"]})
        X = np.column_stack([np.ones(len(g)), g["past size"], g["trailing vol"]])
        out.append(g.assign(unexplained=g["size"] - X @ np.linalg.lstsq(X, g["size"], rcond=None)[0]))
    return pd.concat(out)


rows = []
for h in IDXS:
    d = signals(h)
    w = unexplained_size(d, [f"{m} intensity" for m in scores])
    for m in scores:
        rows.append({"horizon": h, "signal": f"{m} score", "against": "next return", **rank_ic(d, f"{m} score", "return")})
        rows.append({"horizon": h, "signal": f"{m} intensity", "against": "next move size, within instrument",
                     **rank_ic(w, f"{m} intensity", "size")})
        rows.append({"horizon": h, "signal": f"{m} intensity", "against": "beyond past move and trailing volatility",
                     **rank_ic(w, f"{m} intensity", "unexplained")})
pd.DataFrame(rows).set_index(["horizon", "signal", "against"])

### Prediction markets

The 200 highest-volume Polymarket events that closed with an end date in the period; from each, the highest-volume binary market that resolved, was still open 7 days before its end date and had a current price then and 30 days of history (`sbench.markets`). clef and lev give P(Yes) with `threshold_noul` at 0.50, from the market question and its last 60 daily prices; the market's own price at the forecast time is the bar to beat. Downloading needs access to Polymarket's public API.

In [ ]:
PM = SNAPSHOTS / "polymarket"
events_file = PM / "events.json"
if not events_file.exists():
    PM.mkdir(parents=True, exist_ok=True)
    events_file.write_text(json.dumps(fetch_polymarket_closed_events(START, END)), encoding="utf-8")
EVENTS = json.loads(events_file.read_text(encoding="utf-8"))
QUESTIONS = markets.select_markets(
    EVENTS,
    history=lambda tok: cached(PM / "history" / f"{tok}.csv", load_polymarket_history, tok),
    recent=lambda tok, a, b: cached(PM / "recent" / f"{tok}_{a:%Y%m%d%H}.csv", load_polymarket_history, tok, 60,
                                    start=a, end=b))
print(f"{len(QUESTIONS)} markets from {len(EVENTS)} events; resolved Yes: {np.mean([q.outcome for q in QUESTIONS]):.0%}")

PM_PREDICTORS = [DecisionModelPredictor(m, ThresholdNoul(), STATES["price"], name=f"{m}/threshold_noul")
                 for m in ("clef", "lev") if m in SERVED]
answers = markets.ask_all(PM_PREDICTORS, QUESTIONS, RESULTS / "markets.jsonl")
pm = markets.per_question(answers, QUESTIONS)
display(pm.groupby("predictor")["brier"].agg(["mean", "size"]))
skill_table(pm, "brier", baseline=markets.MARKET).set_index("predictor")

## Deviations

_Written after the run._ Every difference from the plan or from this notebook as committed before the period, with its reason.

## Conclusions

_Written after the run._